``get_labels_tiered`` keeps the **positive set fixed** (``targets >= Q(q_pos)``) and sweeps **stepwise-lower negative cuts** (``targets <= Q(q_neg)``), dropping the middle band each tier. Pass the value source and each tier returns its row-matched ``df_parts`` / ``dict_num_parts`` subset plus the binary labels:

In [1]:
import numpy as np
import pandas as pd
import aaanalysis as aa

targets = np.linspace(0, 1, 40)
df_parts = pd.DataFrame({"tmd": [f"AC{i:02d}" for i in range(40)]})
tiers = aa.SequenceFeature.get_labels_tiered(targets, q_pos=0.8, list_q_neg=[0.8, 0.5, 0.3], df_parts=df_parts)
{q: (len(dfp), int(y.sum())) for q, (dfp, _, y) in tiers.items()}  # q_neg -> (n_selected, n_positive)

{0.8: (40, 8), 0.5: (28, 8), 0.3: (20, 8)}

Like OvO, each tier already carries its row-matched ``df_parts``, so build a ``CPP`` per tier directly:

In [2]:
df_seq = aa.load_dataset(name="DOM_GSEC", n=12)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
targets = np.linspace(0, 1, len(df_parts))

for q_neg, (df_tier, _, binary) in aa.SequenceFeature.get_labels_tiered(
        targets, q_pos=0.7, list_q_neg=[0.5, 0.3], df_parts=df_parts).items():
    df_feat = aa.CPP(df_parts=df_tier).run(labels=binary, n_filter=3)
    print(f"q_neg={q_neg}: {len(df_tier)} samples -> {len(df_feat)} features")

1. CPP creates 580140 features for 19 samples
1.1 Assigning scale values to parts
   |                         | 0.0%

   |........                 | 33.3%

   |................         | 66.7%

   |.........................| 100.0%


1.2 Streaming pre-filter stats (mask in stream)
   |                         | 0.0%

...

   |.........................| 100.0%


1.2 Streaming pre-filter stats (mask in stream)
   |                         | 0.0%

   |.........................| 100.0%

   |.........................| 100.0%
2. CPP pre-filters 29007 features (5.0%) with highest 'abs_mean_dif' and 'max_std_test' <= 0.2 (kept=520556 of 580140)


3. CPP filtering algorithm
4. CPP returns df of 3 unique features with general information and statistics
q_neg=0.3: 14 samples -> 3 features


**What can go wrong?** A tier that leaves no negatives (or constant targets) raises (value source supplied so the check is reached):

In [3]:
try:
    df_parts = pd.DataFrame({"tmd": list("ACGT")})
    aa.SequenceFeature.get_labels_tiered([5.0, 5.0, 5.0, 5.0], q_pos=0.8, list_q_neg=[0.3], df_parts=df_parts)
except ValueError as e:
    print("ValueError:", e)

ValueError: tier q_neg=0.3 yields a single class (n_pos=4, n_neg=0); choose q_pos/q_neg that keep both groups non-empty.


**Further parameters.** As for OvO, a numerical ``dict_num_parts`` value source is subset per tier, and ``label_test`` / ``label_ref`` set the positive / reference label values:

In [4]:
targets_ex = np.linspace(0, 1, 8)
df_parts_ex = pd.DataFrame({"tmd": [f"AC{i:02d}" for i in range(8)]})
dict_num_parts = {"tmd": np.random.default_rng(0).random((8, 4, 2))}
tiers = aa.SequenceFeature.get_labels_tiered(targets_ex, q_pos=0.7, list_q_neg=[0.5, 0.3],
                                             df_parts=df_parts_ex, dict_num_parts=dict_num_parts,
                                             label_test=1, label_ref=0)
{q: (dnp["tmd"].shape, int(y.sum())) for q, (_, dnp, y) in tiers.items()}

{0.5: ((7, 4, 2), 3), 0.3: ((6, 4, 2), 3)}